# Agent Framework Benchmark — Results Analysis

This notebook loads benchmark results and creates interactive Plotly charts for comparison.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Load results
results_path = Path("../results/benchmark_results.csv")
if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} results")
    display(df.head())
else:
    df = None
    print(f"No results found at {results_path}")
    print("Run the benchmark first: uv run python -m benchmark.runner")
    print("Subsequent cells will be skipped.")

Loaded 27 results


,framework,company,iteration,completeness,accuracy,structure,insight,readability,overall,reasoning,latency_seconds,input_tokens,output_tokens,total_tokens,estimated_cost,report_text
0,langgraph,Anthropic,1,9.0,8.0,10.0,9.0,10.0,9.2,"The report is comprehensive, well-structured, ...",1355.36,3894,5773,9667,0.0,# Research Report: Anthropic \n\n---\n\n## **...
1,langgraph,Anthropic,2,10.0,9.0,10.0,10.0,10.0,9.8,"The report is exceptionally comprehensive, cov...",1514.72,3432,5478,8910,0.0,# Research Report: Anthropic \n\n## Executive...
2,langgraph,Anthropic,3,9.0,8.0,9.0,9.0,9.0,8.8,"The report is comprehensive, well-structured, ...",1474.16,3432,5478,8910,0.0,# Research Report: Anthropic \n\n## Executive...
3,langgraph,Stripe,1,9.0,10.0,10.0,9.0,10.0,9.6,"The report is comprehensive, accurate, and wel...",1184.77,4045,5428,9473,0.0,**Research Report: Stripe** \n\n---\n\n### **...
4,langgraph,Stripe,2,10.0,9.0,10.0,10.0,10.0,9.8,"The report is exceptionally comprehensive, cov...",870.82,4045,5193,9238,0.0,**Research Report: Stripe** \n\n---\n\n### **...


## Quality Scores by Framework

In [2]:
if df is not None:
    # Grouped bar chart: quality scores per framework
    quality_cols = ["completeness", "accuracy", "structure", "insight", "readability", "overall"]
    quality_avg = df.groupby("framework")[quality_cols].mean().reset_index()
    quality_melted = quality_avg.melt(id_vars="framework", var_name="metric", value_name="score")

    fig = px.bar(
        quality_melted,
        x="framework",
        y="score",
        color="metric",
        barmode="group",
        title="Average Quality Scores by Framework",
        labels={"score": "Score (1-10)", "framework": "Framework"},
    )
    fig.update_layout(yaxis_range=[0, 10])
    fig.show()

## Latency Comparison

In [3]:
if df is not None:
    # Bar chart: average latency per framework
    latency_avg = df.groupby("framework")["latency_seconds"].mean().reset_index()
    latency_avg = latency_avg.sort_values("latency_seconds")

    fig = px.bar(
        latency_avg,
        x="framework",
        y="latency_seconds",
        title="Average Latency by Framework",
        labels={"latency_seconds": "Latency (seconds)", "framework": "Framework"},
        color="framework",
    )
    fig.show()

## Token Usage

In [4]:
if df is not None:
    # Bar chart: average token usage per framework
    token_avg = df.groupby("framework")[["input_tokens", "output_tokens"]].mean().reset_index()
    token_melted = token_avg.melt(id_vars="framework", var_name="type", value_name="tokens")

    fig = px.bar(
        token_melted,
        x="framework",
        y="tokens",
        color="type",
        barmode="stack",
        title="Average Token Usage by Framework",
        labels={"tokens": "Tokens", "framework": "Framework"},
    )
    fig.show()

## Cost Comparison

In [5]:
if df is not None:
    # Table: cost comparison
    cost_summary = df.groupby("framework").agg({
        "estimated_cost": ["mean", "sum"],
        "total_tokens": "mean",
        "latency_seconds": "mean",
        "overall": "mean",
    }).round(4)

    cost_summary.columns = ["Avg Cost ($)", "Total Cost ($)", "Avg Tokens", "Avg Latency (s)", "Avg Quality"]
    display(cost_summary)

,Avg Cost ($),Total Cost ($),Avg Tokens,Avg Latency (s),Avg Quality
framework,,,,,
agents_sdk,0.0,0.0,0.0000,887.4022,9.6889
autogen,0.0,0.0,0.0000,0.4578,8.7222
langgraph,0.0,0.0,9238.7778,1248.5456,9.6111


## Score Distribution (Box Plots)

In [6]:
if df is not None:
    # Box plots: score distribution per framework
    fig = px.box(
        df,
        x="framework",
        y="overall",
        color="framework",
        title="Overall Quality Score Distribution by Framework",
        labels={"overall": "Overall Score (1-10)", "framework": "Framework"},
        points="all",
    )
    fig.update_layout(yaxis_range=[0, 10])
    fig.show()

In [7]:
if df is not None:
    # Box plots for all quality metrics
    quality_df = df.melt(
        id_vars=["framework"],
        value_vars=quality_cols,
        var_name="metric",
        value_name="score",
    )

    fig = px.box(
        quality_df,
        x="metric",
        y="score",
        color="framework",
        title="Quality Metric Distributions by Framework",
        labels={"score": "Score (1-10)", "metric": "Metric"},
    )
    fig.update_layout(yaxis_range=[0, 10])
    fig.show()

## Summary Table

In [8]:
if df is not None:
    # Comprehensive summary table
    summary = df.groupby("framework").agg({
        "overall": "mean",
        "completeness": "mean",
        "accuracy": "mean",
        "structure": "mean",
        "insight": "mean",
        "readability": "mean",
        "latency_seconds": "mean",
        "total_tokens": "mean",
        "estimated_cost": "sum",
    }).round(2)

    summary.columns = [
        "Quality", "Completeness", "Accuracy", "Structure",
        "Insight", "Readability", "Latency (s)", "Tokens", "Total Cost ($)",
    ]

    # Rank by overall quality
    summary = summary.sort_values("Quality", ascending=False)
    display(summary)

,Quality,Completeness,Accuracy,Structure,Insight,Readability,Latency (s),Tokens,Total Cost ($)
framework,,,,,,,,,
agents_sdk,9.69,9.56,9.67,9.89,9.56,9.78,887.40,0.00,0.0
langgraph,9.61,9.56,9.22,9.89,9.56,9.89,1248.55,9238.78,0.0
autogen,8.72,9.56,9.89,8.78,7.00,9.22,0.46,0.00,0.0


## Per-Company Quality Breakdown

In [ ]:
if df is not None:
    # Faceted bar chart: quality per company per framework
    company_quality = df.groupby(["framework", "company"])["overall"].agg(["mean", "std"]).reset_index()
    company_quality.columns = ["framework", "company", "mean_quality", "std_quality"]

    fig = px.bar(
        company_quality,
        x="company",
        y="mean_quality",
        color="framework",
        barmode="group",
        error_y="std_quality",
        title="Average Quality Score by Company and Framework",
        labels={"mean_quality": "Overall Score (1-10)", "company": "Company"},
    )
    fig.update_layout(yaxis_range=[0, 10])
    fig.show()

## Consistency Analysis (Standard Deviation Across Iterations)

In [ ]:
if df is not None:
    # Consistency: lower std = more consistent results
    consistency = df.groupby(["framework", "company"]).agg({
        "overall": ["mean", "std"],
        "latency_seconds": ["mean", "std"],
    }).round(2)
    consistency.columns = ["Avg Quality", "Quality Std", "Avg Latency", "Latency Std"]
    consistency = consistency.reset_index()

    print("Consistency across iterations (lower std = more consistent):")
    display(consistency)

    # Framework-level consistency
    fw_consistency = df.groupby("framework")["overall"].agg(["mean", "std", "min", "max"]).round(2)
    fw_consistency.columns = ["Mean", "Std Dev", "Min", "Max"]
    fw_consistency["Range"] = fw_consistency["Max"] - fw_consistency["Min"]
    print("\nFramework-level quality consistency:")
    display(fw_consistency.sort_values("Std Dev"))

## Quality vs Latency Trade-off

In [ ]:
if df is not None:
    # Scatter plot: quality vs latency (each dot = one run)
    fig = px.scatter(
        df,
        x="latency_seconds",
        y="overall",
        color="framework",
        symbol="company",
        size="total_tokens",
        size_max=20,
        title="Quality vs Latency Trade-off (size = token count)",
        labels={
            "latency_seconds": "Latency (seconds)",
            "overall": "Overall Quality Score",
        },
        hover_data=["company", "iteration"],
    )
    fig.update_layout(yaxis_range=[0, 10])
    fig.show()

## Radar Chart — Framework Quality Profiles

In [ ]:
if df is not None:
    # Radar chart comparing quality dimensions per framework
    radar_cols = ["completeness", "accuracy", "structure", "insight", "readability"]
    radar_avg = df.groupby("framework")[radar_cols].mean()

    fig = go.Figure()
    for fw in radar_avg.index:
        values = radar_avg.loc[fw].tolist()
        values.append(values[0])  # close the polygon
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=radar_cols + [radar_cols[0]],
            fill="toself",
            name=fw,
            opacity=0.6,
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
        title="Quality Dimension Profiles by Framework",
        showlegend=True,
    )
    fig.show()

## Report Length Analysis

In [ ]:
if df is not None:
    # Report length: word count and character count
    df["report_chars"] = df["report_text"].str.len()
    df["report_words"] = df["report_text"].str.split().str.len()

    fig = px.box(
        df,
        x="framework",
        y="report_words",
        color="framework",
        title="Report Length (Word Count) by Framework",
        labels={"report_words": "Word Count", "framework": "Framework"},
        points="all",
    )
    fig.show()

    # Report length vs quality correlation
    fig2 = px.scatter(
        df,
        x="report_words",
        y="overall",
        color="framework",
        title="Report Length vs Quality Score",
        labels={"report_words": "Word Count", "overall": "Overall Quality"},
        hover_data=["company", "iteration"],
        trendline="ols",
    )
    fig2.update_layout(yaxis_range=[0, 10])
    fig2.show()

    print(f"\nReport length statistics:")
    display(df.groupby("framework")[["report_words", "report_chars"]].describe().round(0))

## Statistical Significance Testing

Are the quality differences between frameworks statistically significant, or just noise?
We use the Kruskal-Wallis H-test (non-parametric, doesn't assume normality) followed by
pairwise Mann-Whitney U tests with Bonferroni correction.

In [ ]:
if df is not None:
    from scipy import stats
    from itertools import combinations

    frameworks = df["framework"].unique()
    groups = [df[df["framework"] == fw]["overall"].values for fw in frameworks]

    # Step 1: Kruskal-Wallis H-test (are ANY groups different?)
    h_stat, kw_p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H-test: H={h_stat:.3f}, p={kw_p:.4f}")
    if kw_p < 0.05:
        print("  -> Significant difference exists between at least two frameworks (p < 0.05)")
    else:
        print("  -> No significant difference found between frameworks (p >= 0.05)")

    # Step 2: Pairwise Mann-Whitney U tests with Bonferroni correction
    n_comparisons = len(list(combinations(frameworks, 2)))
    alpha_corrected = 0.05 / n_comparisons

    print(f"\nPairwise Mann-Whitney U tests (Bonferroni-corrected alpha = {alpha_corrected:.4f}):")
    print(f"{'Pair':<35} {'U-stat':>8} {'p-value':>10} {'Significant?':>14} {'Effect size r':>14}")
    print("-" * 85)

    for fw_a, fw_b in combinations(frameworks, 2):
        a = df[df["framework"] == fw_a]["overall"].values
        b = df[df["framework"] == fw_b]["overall"].values
        u_stat, p_val = stats.mannwhitneyu(a, b, alternative="two-sided")
        # Effect size r = Z / sqrt(N)
        z_stat = stats.norm.ppf(1 - p_val / 2) if p_val < 1.0 else 0
        r_effect = z_stat / (len(a) + len(b)) ** 0.5
        sig = "YES" if p_val < alpha_corrected else "no"
        print(f"  {fw_a} vs {fw_b:<20} {u_stat:>8.1f} {p_val:>10.4f} {sig:>14} {r_effect:>14.3f}")

    print(f"\nEffect size interpretation: r < 0.3 = small, 0.3-0.5 = medium, > 0.5 = large")

    # Step 3: Also test latency differences
    print(f"\n--- Latency significance ---")
    lat_groups = [df[df["framework"] == fw]["latency_seconds"].values for fw in frameworks]
    h_lat, kw_lat_p = stats.kruskal(*lat_groups)
    print(f"Kruskal-Wallis H-test (latency): H={h_lat:.3f}, p={kw_lat_p:.6f}")
    if kw_lat_p < 0.05:
        print("  -> Significant latency difference exists between frameworks")

## Token Efficiency

Quality-per-token measures how much value each framework extracts from its LLM calls.
Higher is better — achieving the same quality with fewer tokens means lower cost in production.

In [ ]:
if df is not None:
    # Only compute for frameworks that actually tracked tokens
    df_with_tokens = df[df["total_tokens"] > 0].copy()

    if len(df_with_tokens) > 0:
        # Quality per 1K tokens
        df_with_tokens["quality_per_1k_tokens"] = (
            df_with_tokens["overall"] / df_with_tokens["total_tokens"] * 1000
        )
        # Quality per second of latency
        df_with_tokens["quality_per_second"] = (
            df_with_tokens["overall"] / df_with_tokens["latency_seconds"]
        )

        efficiency = df_with_tokens.groupby("framework").agg({
            "quality_per_1k_tokens": "mean",
            "quality_per_second": "mean",
            "total_tokens": "mean",
            "overall": "mean",
            "latency_seconds": "mean",
        }).round(4)
        efficiency.columns = [
            "Quality / 1K Tokens", "Quality / Second",
            "Avg Tokens", "Avg Quality", "Avg Latency (s)",
        ]
        efficiency = efficiency.sort_values("Quality / 1K Tokens", ascending=False)

        print("Token Efficiency (higher = better value per token):")
        display(efficiency)

        fig = px.bar(
            df_with_tokens,
            x="framework",
            y="quality_per_1k_tokens",
            color="framework",
            title="Token Efficiency: Quality Score per 1K Tokens (higher = better)",
            labels={"quality_per_1k_tokens": "Quality / 1K Tokens", "framework": "Framework"},
        )
        fig.show()
    else:
        print("No frameworks with token tracking data available.")
        print("Token efficiency analysis requires total_tokens > 0.")

## Latency Breakdown Heatmap

Shows average latency for each framework-company pair.
Reveals whether certain research targets are inherently harder (more data to process,
more complex to analyze) regardless of framework choice.

In [ ]:
if df is not None:
    import numpy as np

    # Pivot: framework (rows) x company (columns), values = avg latency
    latency_pivot = df.pivot_table(
        values="latency_seconds",
        index="framework",
        columns="company",
        aggfunc="mean",
    ).round(1)

    # Also create a quality pivot for a side-by-side heatmap
    quality_pivot = df.pivot_table(
        values="overall",
        index="framework",
        columns="company",
        aggfunc="mean",
    ).round(2)

    # Latency heatmap
    fig = px.imshow(
        latency_pivot,
        text_auto=True,
        color_continuous_scale="YlOrRd",
        title="Average Latency (seconds) — Framework x Company",
        labels=dict(x="Company", y="Framework", color="Latency (s)"),
        aspect="auto",
    )
    fig.show()

    # Quality heatmap
    fig2 = px.imshow(
        quality_pivot,
        text_auto=True,
        color_continuous_scale="RdYlGn",
        title="Average Quality Score — Framework x Company",
        labels=dict(x="Company", y="Framework", color="Quality"),
        aspect="auto",
        zmin=7, zmax=10,
    )
    fig2.show()

    # Which company is hardest/easiest across all frameworks?
    print("Company difficulty (avg latency across all frameworks):")
    company_difficulty = df.groupby("company").agg({
        "latency_seconds": "mean",
        "overall": "mean",
    }).round(1)
    company_difficulty.columns = ["Avg Latency (s)", "Avg Quality"]
    display(company_difficulty.sort_values("Avg Latency (s)", ascending=False))